# Model Test

In [1]:
DATAROOT = "/SSD2/bgkang/Chemomile/"

MODELPATH = dict(
    FP = "../Model/FP-2024-12-17-01-38-53", # FP
    AIT = "../Model/AIT-2024-12-17-01-44-45", # AIT
    FLVL = "../Model/FLVL-2024-12-17-01-56-39", # FLVL
    FLVU = "../Model/FLVU-2024-12-17-02-03-26", # FLVU
    HCOM = "../Model/HCOM-2024-12-17-01-48-28", # HCOM
    ESOL = "../Model/ESOL-2024-12-17-01-31-49", # ESOL
)

parameters = {
    'default' : dict(
        subfrag_size = 10,
        edge_size = 3,
        out_size = 1,
        seed = 1217,
        batch_size = 128,
        max_epoch = 200,
        verbose = False,
        save = False,
    ),
    'FP' : {'hidden_size' : 128, 'dropout' : 0.276, 'num_layers' : 6, 'num_timesteps' : 6, 'lr_init' : 0.001, 'gamma' : 0.978, 'weight_decay' : 6.62E-3},
    'AIT' : {'hidden_size' : 64, 'dropout' : 0.220, 'num_layers' : 3, 'num_timesteps' : 6, 'lr_init' : 0.001, 'gamma' : 0.981, 'weight_decay' : 2.67E-3},
    'HCOM' : {'hidden_size' : 128, 'dropout' : 0.230, 'num_layers' : 6, 'num_timesteps' : 5, 'lr_init' : 0.0001, 'gamma' : 0.979, 'weight_decay' : 3.92E-4},
    'FLVL' : {'hidden_size' : 128, 'dropout' : 0.450, 'num_layers' : 5, 'num_timesteps' : 6, 'lr_init' : 0.001, 'gamma' : 0.984, 'weight_decay' : 6.95E-4},
    'FLVU' : {'hidden_size' : 256, 'dropout' : 0.307, 'num_layers' : 5, 'num_timesteps' : 7, 'lr_init' : 0.001, 'gamma' : 0.983, 'weight_decay' : 1.26E-2},
    'ESOL' : {'hidden_size' : 32, 'dropout' : 0.238, 'num_layers' : 4, 'num_timesteps' : 6, 'lr_init' : 0.0001, 'gamma' : 0.993, 'weight_decay' : 1.23E-3},
}

results = {
    'FP' : {
        'test' : {
            'true' : [],
            'pred' : [],
        },
        'total' : {
            'true' : [],
            'pred' : [],
        },
    },
    'AIT' : {
        'test' : {
            'true' : [],
            'pred' : [],
        },
        'total' : {
            'true' : [],
            'pred' : [],
        },
    },
    'FLVL' : {
        'test' : {
            'true' : [],
            'pred' : [],
        },
        'total' : {
            'true' : [],
            'pred' : [],
        },
    },
    'FLVU' : {
        'test' : {
            'true' : [],
            'pred' : [],
        },
        'total' : {
            'true' : [],
            'pred' : [],
        },
    },
    'HCOM' : {
        'test' : {
            'true' : [],
            'pred' : [],
        },
        'total' : {
            'true' : [],
            'pred' : [],
        },
    },
}

In [2]:
import numpy as np
for target in results:
    test = np.genfromtxt(f'./result/{target}_test.out')
    results[target]['test']['true'] = test[:,0]
    results[target]['test']['pred'] = test[:,1]

    total = np.genfromtxt(f'./result/{target}_total.out')
    results[target]['total']['true'] = total[:,0]
    results[target]['total']['pred'] = total[:,1]

In [ ]:
import sys
import torch
sys.path.append("/SSD2/bgkang/Chemomile")
from src.data import Dataset
from src.model import Chemomile
from src.train import Training

"""
for key in results.keys():
    model = Chemomile(
        subfrag_size = parameters['default']['subfrag_size'],
        hidden_size = parameters[key]['hidden_size'],
        out_size = parameters['default']['out_size'],
        edge_size = parameters['default']['edge_size'],
        dropout = parameters[key]['dropout'],
        num_layers = parameters[key]['num_layers'],
        num_timesteps = parameters[key]['num_timesteps'],
    )
    model.load_state_dict(torch.load(MODELPATH[key]))

    dataset = Dataset(
        target = key,
        root = DATAROOT,
        seed = parameters['default']['seed'],
        batch_size = parameters['default']['batch_size'],
    )

    training = Training(
        model = model, 
        dataset = dataset, 
        parameters = parameters[key] | parameters['default'],
    )

    results[key]['test']['true'], results[key]['test']['pred'] = training.eval(total = False)
    results[key]['total']['true'], results[key]['total']['pred'] = training.eval(total = True)

print("All results are updated! >:D")
"""

# Parity Plot

In [4]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import r2_score

COLORMAP = {
    'FP' : "#AF272F",
    'AIT' : "#00558C",
    'FLVL' : "#D38235",
    'FLVU' : "#642F6C",
    'HCOM' : "#719949",
}

def metric(true, pred):
    mae = np.abs(true - pred).mean()
    rmse = np.sqrt(np.power((true - pred), 2.0).mean())
    mape = np.abs((true - pred) / true * 100).mean()
    r2 = r2_score(true, pred)
    return mae, rmse, mape, r2
    

def beautifulTPPlot(top, bottom, key, unit):
    top.spines['top'].set_visible(False)
    top.spines['right'].set_visible(False)
    top.spines['left'].set_linewidth(5)
    top.spines['bottom'].set_linewidth(5)
    top.tick_params(axis = 'both', width = 5, length = 5, direction = 'in', labelsize = 50, pad = 10)
    top.set_xlabel("True %s" % (unit), fontsize = 75)
    top.set_ylabel("Predicted %s" % (unit), fontsize = 75)

    combined = np.stack([results[key]['total']['true'], results[key]['total']['pred']])

    top.plot(
        [combined.min(), combined.max()],
        [combined.min(), combined.max()],
        linestyle = '--', color = 'k', linewidth = 5)
    top.plot(
        results[key]['test']['true'],
        results[key]['test']['pred'],
        marker = 'o', markersize = 30, markeredgewidth = 7, markerfacecolor = "#FFFFFF", markeredgecolor = COLORMAP[key], linestyle = 'None', alpha = 0.8
    )
    """
    top.annotate(
        "•MAE: %.03f\n•RMSE: %.03f\n•MAPE: %.03f\n•R$^2$: %.03f" % metric(results[key]['test']['true'], results[key]['test']['pred']),
        xy = (0.1, 0.7),
        xycoords = 'axes fraction',
        fontsize = 30,
        va = 'bottom'
    )
    """
    top.annotate(
        "$R^2$: %.3f" % metric(results[key]['test']['true'], results[key]['test']['pred'])[-1],
        xy = (0.1, 0.7),
        xycoords = 'axes fraction',
        fontsize = 70,
        va = 'bottom'
    )
    
    bottom.spines['top'].set_visible(False)
    bottom.spines['right'].set_visible(False)
    bottom.spines['left'].set_linewidth(5)
    bottom.spines['bottom'].set_linewidth(5)
    bottom.tick_params(width = 5, length = 10, direction = 'in')
    bottom.tick_params(axis = 'both', width = 5, length = 5, direction = 'in', labelsize = 50, pad = 10)
    bottom.set_xlabel("True %s" % (unit), fontsize = 75)
    bottom.set_ylabel("Predicted %s" % (unit), fontsize = 75)
    
    bottom.plot(
        [combined.min(), combined.max()],
        [combined.min(), combined.max()],
        linestyle = '--', color = 'k', linewidth = 5)
    bottom.plot(
        results[key]['total']['true'],
        results[key]['total']['pred'],
        marker = 'o', markersize = 30, markeredgewidth = 7, markerfacecolor = "#FFFFFF", markeredgecolor = COLORMAP[key], linestyle = 'None', alpha = 0.8
    )

    """
    bottom.annotate(
        "•MAE: %.03f\n•RMSE: %.03f\n•MAPE: %.03f\n•R$^2$: %.03f" % metric(results[key]['total']['true'], results[key]['total']['pred']),
        xy = (0.1, 0.7),
        xycoords = 'axes fraction',
        fontsize = 30,
        va = 'bottom',
    )
    """
    bottom.annotate(
        "$R^2$: %.3f" % metric(results[key]['total']['true'], results[key]['total']['pred'])[-1],
        xy = (0.1, 0.7),
        xycoords = 'axes fraction',
        fontsize = 70,
        va = 'bottom'
    )

In [ ]:
fig, ax = plt.subplots(2, 5, figsize = (72, 27), dpi = 300)

beautifulTPPlot(ax[0][0], ax[1][0], "FP", r"""$T_F$
(K)""",)
beautifulTPPlot(ax[0][1], ax[1][1], "AIT", r"""$T_A$
(K)""")
beautifulTPPlot(ax[0][2], ax[1][2], "HCOM", r"""$\Delta H_{com}$
($\times$ 10$^3$ kJ mol$^{-1}$)""")
beautifulTPPlot(ax[0][3], ax[1][3], "FLVL", r"""$\phi_L$
(Vol % in air)""")
beautifulTPPlot(ax[0][4], ax[1][4], "FLVU", r"""$\phi_U$
(Vol % in air)""")

ax[0][0].set_xticks([200, 300, 400, 500])
ax[1][0].set_xticks([200, 300, 400, 500])
ax[0][1].set_xticks([300, 600, 900, 1200])
ax[1][1].set_xticks([300, 600, 900, 1200])
ax[0][2].set_xticks([-18, -12, -6, 0])
ax[1][2].set_xticks([-18, -12, -6, 0])
ax[0][3].set_xticks([0, 5, 10, 15])
ax[1][3].set_xticks([0, 5, 10, 15])
ax[0][4].set_xticks([0, 30, 60, 90])
ax[1][4].set_xticks([0, 30, 60, 90])

ax[0][0].set_yticks([200, 300, 400, 500])
ax[1][0].set_yticks([200, 300, 400, 500])
ax[0][1].set_yticks([300, 600, 900, 1200])
ax[1][1].set_yticks([300, 600, 900, 1200])
ax[0][2].set_yticks([-18, -12, -6, 0])
ax[1][2].set_yticks([-18, -12, -6, 0])
ax[0][3].set_yticks([0, 5, 10, 15])
ax[1][3].set_yticks([0, 5, 10, 15])
ax[0][4].set_yticks([0, 30, 60, 90])
ax[1][4].set_yticks([0, 30, 60, 90])

plt.tight_layout()
plt.savefig('Chemomile_Result.png', dpi = 300)
plt.show()

In [ ]:
fig, ax = plt.subplots(5, 2, figsize = (27, 72), dpi = 300)

beautifulTPPlot(ax[0][0], ax[0][1], "FP", r"""$T_F$
(K)""",)
beautifulTPPlot(ax[1][0], ax[1][1], "AIT", r"""$T_A$
(K)""")
beautifulTPPlot(ax[2][0], ax[2][1], "HCOM", r"""$\Delta H_{com}$
($\times$ 10$^3$ kJ mol$^{-1}$)""")
beautifulTPPlot(ax[3][0], ax[3][1], "FLVL", r"""$\phi_L$
(Vol % in air)""")
beautifulTPPlot(ax[4][0], ax[4][1], "FLVU", r"""$\phi_U$
(Vol % in air)""")

plt.tight_layout()
plt.savefig('Chemomile_Result_portrait.png', dpi = 300)
plt.show()